# Notebook 4: Memory and Multi-step Tasks

## Why

Each run starts cold. That's fine for one question, but a research assistant should remember what you told it an hour ago, or even three steps ago, if not it is not an assistant.

Memory is what lets an agent carry context across steps and across sessions.

## What we'll build
1. **Short-term memory**
2. **Long-term memory**
3. **A multi-step task** that only works if the agent remembers an earlier step.


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ms-cc-org/AGENTIC-AI-Workshop/blob/main/notebooks/03_research_assistant.ipynb)

# STEP 1 - Setup

In [ ]:
#  SETUP cell. This is for a provider-agnostic LLM client.
#  Read this cell; every notebook uses this same adapter.
#  Switch providers by changing PROVIDER.
#  Nothingelse in the notebook changes. an agent is a pattern, not a vendor.
#  In this notebook, we install embedding and numeric packages here

# In Google Colab this cell installs the packages. Locally, run once.

%pip install -q anthropic

import os, json

PROVIDER = "mock"   # "anthropic" | "mock"
                      # "mock" runs this notebook with NO API key, 

MODELS = {"anthropic": "claude-haiku-4-5-20251001"}  # cheapest current Claude model


if PROVIDER == "anthropic":
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

# Normalized shapes we use everywhere (so the agent code never mentions a vendor):
#   message: {"role":"user","content":str}
#            {"role":"assistant","content":str|None,"tool_calls":[{id,name,args}]}
#            {"role":"tool","tool_call_id":id,"name":name,"content":str}
#   tool:    {"name":str,"description":str,"parameters":<json-schema>}
#   reply:   {"text":str,"tool_calls":[{id,name,args}],"stop_reason":str}

_MOCK_SCRIPT = []
def mock_reset(script):
    global _MOCK_SCRIPT; _MOCK_SCRIPT = list(script)
def _mock_call(messages, tools):
    if _MOCK_SCRIPT:
        kind, payload = _MOCK_SCRIPT.pop(0)
        if kind == "tool":
            return {"text":"", "tool_calls":[{"id":"mock_"+payload["name"],
                    "name":payload["name"], "args":payload["args"]}], "stop_reason":"tool_use"}
        return {"text":payload, "tool_calls":[], "stop_reason":"end"}
    last = next((m for m in reversed(messages) if m["role"] in ("user","tool")), {"content":""})
    return {"text": f"[mock reply to] {str(last.get('content',''))[:80]}",
            "tool_calls":[], "stop_reason":"end"}


def _to_anthropic(messages):
    out=[]
    for m in messages:
        if m["role"]=="user":
            out.append({"role":"user","content":m["content"]})
        elif m["role"]=="assistant":
            blocks=[]
            if m.get("content"): blocks.append({"type":"text","text":m["content"]})
            for tc in m.get("tool_calls",[]):
                blocks.append({"type":"tool_use","id":tc["id"],"name":tc["name"],"input":tc["args"]})
            out.append({"role":"assistant","content":blocks})
        elif m["role"]=="tool":
            out.append({"role":"user","content":[{"type":"tool_result",
                        "tool_use_id":m["tool_call_id"],"content":str(m["content"])}]})
    return out

def call_llm(messages, tools=None, system=None, max_tokens=1024, temperature=0):
    '''One function. Any provider. This is portable across whatever API key you have or your institution has.'''
    if PROVIDER=="mock":
        return _mock_call(messages, tools)
    if PROVIDER=="anthropic":
        from anthropic import Anthropic
        client=Anthropic()
        kw=dict(model=MODELS["anthropic"], max_tokens=max_tokens,
                temperature=temperature, messages=_to_anthropic(messages))
        if system: kw["system"]=system
        if tools: kw["tools"]=[{"name":t["name"],"description":t["description"],
                                "input_schema":t["parameters"]} for t in tools]
        r=client.messages.create(**kw)
        text=""; calls=[]
        for b in r.content:
            if b.type=="text": text+=b.text
            elif b.type=="tool_use": calls.append({"id":b.id,"name":b.name,"args":b.input})
        return {"text":text,"tool_calls":calls,"stop_reason":r.stop_reason}
    raise ValueError("PROVIDER: 'anthropic' or 'mock'; got" +PROVIDER)

print(f"Setup ready. PROVIDER={PROVIDER!r}. Commercial provider: Anthropic only.")

# From Notebook 2

Notebook 2 gives us a map where a tool fills gaps for a model by giving capabilities needs to solve tasks. 

In [ ]:
def run_agent(user_task, tools_spec, tool_fns, system=None, max_steps=6, verbose=True):
    '''The agent loop. The idea is:
       plan -> act (maybe call a tool) -> observe (feed the result back) -> repeat,
       until the model returns a final answer instead of a tool call.'''
    messages = [{"role":"user","content":user_task}]
    for step in range(1, max_steps+1):
        reply = call_llm(messages, tools=tools_spec, system=system)
        if reply["tool_calls"]:                                  # the model wants a tool
            messages.append({"role":"assistant","content":reply["text"],
                             "tool_calls":reply["tool_calls"]})
            for tc in reply["tool_calls"]:
                if verbose: print(f"  step {step}: tool `{tc['name']}` <- {tc['args']}")
                try:    result = tool_fns[tc["name"]](**tc["args"])   # run the real function
                except Exception as e: result = f"ERROR: {e}"
                messages.append({"role":"tool","tool_call_id":tc["id"],
                                 "name":tc["name"],"content":result})   # observe
        else:                                                    # no tool -> we are done
            if verbose: print(f"  step {step}: final answer")
            return reply["text"], messages
    return "Stopped: hit max_steps.", messages

# Step 2 - Short-term memory

The simplest memory is just *keeping the transcript* and sending it back every turn. Watch what happens with and without it.

In [ ]:
def chat_with_memory(history, user_msg):
    history = history + [{"role":"user","content":user_msg}]
    reply = call_llm(history)
    history = history + [{"role":"assistant","content":reply["text"]}]
    return history, reply["text"]

# With memory: the agent has the earlier turn in context.
mock_reset([("final","Your project is NSF-AI-InfraXXXXXXXX"),
            ("final","Earlier you told me your project is MS-CC-AI-Enablement Series")])
hist=[]
hist,_ = chat_with_memory(hist, "My project is MS-CC-AI-Enablement Series. Remember that.")
hist, ans = chat_with_memory(hist, "What is my project again?")
print("WITH memory ->", ans)

# Without memory: each call starts fresh, so it cannot know.
mock_reset([("final","I don't have any earlier context about your project.")])
_, ans2 = chat_with_memory([], "What is my project again?")
print("WITHOUT memory ->", ans2)

## Step 3 Long term memory

## Step 4 Semantic memory

## Step 5 - A multi-step task that needs memory

## Exercise 